# Imports e Bibliotecas

In [233]:
import pandas as pd
import numpy as np
from enum import Enum
from math import floor as floor
from math import inf as inf

# Dataset

In [210]:
deep_seek_v3_coleta = pd.read_csv("/content/Corpus_DeepSeek-V3.csv")
gpt_2_coleta = pd.read_csv("/content/Corpus_gpt-2.csv")
grok_1_coleta = pd.read_csv("/content/Corpus_grok-1.csv")
llama_coleta = pd.read_csv("/content/Corpus_llama.csv")
llama3_coleta = pd.read_csv("/content/Corpus_llama3.csv")

In [211]:
deep_seek_v3_coleta['Repositorio'] = 'DeepSeek_V3'
gpt_2_coleta['Repositorio'] = 'GPT_2'
grok_1_coleta['Repositorio'] = 'Grok_1'
llama_coleta['Repositorio'] = 'Llama'
llama3_coleta['Repositorio'] = 'Llama3'

In [212]:
todos = pd.concat([deep_seek_v3_coleta, gpt_2_coleta, grok_1_coleta, llama_coleta, llama3_coleta], ignore_index=True)

## Adicionando coluna $Validadas$
Representa se a *issue* está nos conjuntos das *issues* validadas como relacionadas à ética ou não

In [213]:
id_issues_validadas = pd.read_csv("/content/validadas.csv")

In [214]:
def addColunaValidada(dataset, id_issues_validadas):
  dataset['Validada'] = dataset['IdIssue'].isin(id_issues_validadas['IdIssue'])
  return dataset

In [215]:
todos = addColunaValidada(todos,id_issues_validadas)

## Identificando quem contribuiu para discussão

In [216]:
contribuintes = pd.read_csv('/content/ValidacaoExtra.csv')

In [217]:
contribuintes['Soma_Total'] = contribuintes['QtdComentarios'] + contribuintes['Info']
soma_agrupada = contribuintes.groupby(['IdIssue', 'Autor'])['Soma_Total'].sum().reset_index()
todos = todos.merge(
    soma_agrupada,
    left_on=['IdIssue', 'AutorComentario'],
    right_on=['IdIssue', 'Autor'],
    how='left'
)
todos.rename(columns={'Soma_Total': 'Volume'}, inplace=True)
todos['Volume'] = todos['Volume'].fillna(0)
todos.drop(columns=['Autor'], inplace=True)

## Anonimização

In [218]:
def anonimizaAutores(dataset):
  cp = dataset.copy()
  cp['AutorComentario'] = pd.factorize(cp['AutorComentario'])[0]
  cp['IdIssue'] = pd.factorize(cp['IdIssue'])[0]
  cp = cp.drop(columns=['LinkIssue','NumeroComentario','Comentario','Info'])
  return cp

### Split Dataset
1. Dataset somente com *issues* validadas (observa quem contribuiu ou não)
3. Dataset somente com *issues* invalidadas
---
> Permite avaliar se existe transversalidade fora do contexto de investigação de aspectos éticos (é mais um plus)

> Quero avaliar também a visão geral, ou seja, se um usuário que abordou aspecto ético também abordou outros temas não relacionados a aspectos éticos ao longo dos repositórios

> Vou analisar também o fato de ter participado numa discussão validada, se há a transversalidade

In [219]:
todos_anon=anonimizaAutores(todos)
todos_validadas=todos_anon[todos_anon['Validada']]
todos_invalidadas=todos_anon[todos_anon['Validada']==False]

# Estrutura de Dados

In [220]:
ids_validados = set(todos_validadas['IdIssue'].tolist())

In [221]:
ids_invalidados = set(todos_invalidadas['IdIssue'].tolist())

In [222]:
class Repositorios(Enum):
  DeepSeek_V3 = 1
  Grok_1 = 2
  GPT_2 = 3
  Llama = 4
  Llama3 = 5

In [174]:
#vou usar as validadas, a visualização pode colorir cada usuario com uma cor e o tamanho do no a qtd de comentarios feitos
#interessante montar rede por aspecto abordado ... ?
#o objetivo é ver a transversalidade do discurso

In [223]:
class Issue:
  def __init__(self, id_issue:int, repositorio:Repositorios):
    self.id_issue = id_issue
    self.repositorio = repositorio
    self.relacionada_etica = self.verificaValidada()

  def verificaValidada(self):
    if self.id_issue in ids_validados:
      return True
    else:
      return False

In [224]:
class No_Usuario:
  def __init__(self, id_participante:int, issues:list):
    self.id_participante = id_participante
    self.issues = issues
    self.repositorio_predominante = self.predominancia_repos() #indica o repo que um usuário mais participou ->
                                                                #pode ser bom ter uma lista de todos os repos q participou
    self.volume_discussao_aspectos_eticos = self.discussao_aspectos_eticos()

  def predominancia_repos(self):
    contabiliza_repo = {Repositorios(1).name:0,
                        Repositorios(2).name:0,
                        Repositorios(3).name:0,
                        Repositorios(4).name:0,
                        Repositorios(5).name:0}

    for i in self.issues:
      if i.repositorio.name in contabiliza_repo.keys():
        contabiliza_repo[i.repositorio.name]+=1

    maior = 0
    for i in contabiliza_repo.values():
      if maior < i:
        maior = i

    for chave, valor in contabiliza_repo.items():
      if valor == maior:
        chave_maior = chave

    return chave_maior

  def discussao_aspectos_eticos(self):
    #1.verificar se usuario esta em issue validada -> self.issues.id_issue in ids_validadas
    #2.nas issues validadas do usuario, buscar em todas_validadas o id_participante dele em AutorComentario
    #3.agrupar usuarios filtrados por AutorComentario e somar dado na coluna Volume em todas_validadas pelos agrupamentos
    #4.retornar resultado
    #parece IA, mas fui eu qm escreveu!!!

    ids_issues_usuario = [i.id_issue for i in self.issues]

    df_filtrado = todos_validadas[todos_validadas['IdIssue'].isin(ids_issues_usuario)]

    filtro_autor = df_filtrado['AutorComentario'] == self.id_participante
    df_filtrado = df_filtrado[filtro_autor]

    volume_total = int(df_filtrado['Volume'].sum())

    return volume_total


## Colocando dados num formato usável

In [225]:
issues = []
participantes = []

for row in todos_anon.itertuples(index=False):
  try:
      r = Repositorios[row.Repositorio]
  except KeyError:
      r = None
  issue = Issue(row.IdIssue,r)
  issues.append(issue)
  participante = No_Usuario(row.AutorComentario, issues) #juntar info de diferentes linhas de autorComentario em um unico nodo
  participantes.append(participante)

Como um mesmo usuário pode estar espalhado em várias issues, ele pode representar mais de um nó, o que não é interessante. Por isso, o método abaixo que encontra esses usuários espalhados e mescla suas informaçõe

In [228]:
def localizaMescla_participante_repetido(participantes):
    participantes_unicos = {}

    for participante in participantes:
        id_part = participante.id_participante

        if id_part not in participantes_unicos:
            # Antes de salvar o participante novo, removemos duplicatas internas de suas issues
            issues_limpas = []
            ids_vistos = set()

            for issue in participante.issues:
                if issue.id_issue not in ids_vistos:
                    issues_limpas.append(issue)
                    ids_vistos.add(issue.id_issue)

            # Atualiza a lista de issues do participante apenas com os itens únicos
            participante.issues = issues_limpas

            # Agora sim, salva no dicionário
            participantes_unicos[id_part] = participante

        else:
            # Lógica de mesclagem (agora funcionando sobre uma lista já limpa)
            p_existente = participantes_unicos[id_part]
            p_existente.volume_discussao_aspectos_eticos += participante.volume_discussao_aspectos_eticos

            ids_issues_existentes = {issue.id_issue for issue in p_existente.issues}

            for nova_issue in participante.issues:
                if nova_issue.id_issue not in ids_issues_existentes:
                    p_existente.issues.append(nova_issue)
                    ids_issues_existentes.add(nova_issue.id_issue)

    return list(participantes_unicos.values())

In [229]:
participantes_tratados = localizaMescla_participante_repetido(participantes)
for i in participantes_tratados:
  print('=====================================================================')
  print(f'Id_Participante={i.id_participante}')
  print(f'Volume={i.volume_discussao_aspectos_eticos}')
  print(f'Issues:')
  print('*********************************************************************')
  for j in i.issues:
    print(f'\tId Issue = {j.id_issue}')
    print(f'\tRepositório = {j.repositorio.name}')
    print(f'\tValidada = {j.relacionada_etica}')

  print('*********************************************************************')
  print('=====================================================================')

A saída de streaming foi truncada nas últimas 5000 linhas.
	Id Issue = 370
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 371
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 372
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 373
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 374
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 375
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 376
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 377
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 378
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 379
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 380
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 381
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 382
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 383
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 384
	Repositório = DeepSeek_V3
	Validada = False
	Id Issue = 385
	Repositór

Durante a construção das arestas, percebi o quão caro seria ligar os nós usando somente listas. Um nó se liga a outro se 2 usuários estão participando de uma mesma issue, ou seja, a issue é a aresta.

Se eu colocasse uma lista de usuários em Issue só faria uma dependência circular que queria evitar

Voltando, para popular a matriz de adjacência, seria preciso olhar o id da issue de um usuário e buscar na lista de issues de todos os outros usuários esse mesmo id para todas as issues. Mesmo que eu marcasse as issues já pesquisadas, isso ainda seria caro. Considerando todas as issues com um tamanho $x$ e $n$ a quantidade de usuários (nós), buscar por um id de issue linearmente seria demandaria uma quantidade absurda de operações.

Por isso, pensei em usar uma ED auxiliar para reduzir o custo dessa busca, tabela hash poderia ser complicado desenvolver uma função hash decente num tempo ábil. Pensei em árvore auto-balanceável pelo custo da busca no pior caso ser $O(log(n))$ e não seria preciso ordenar a entrada. Mas, um heap além de ser mais simples de implementar neste cenário, não perde em complexidade para a árvore, além de ser usado em algoritmos que atuam sobre grafos como Dijkstra

## Heap

In [240]:
class Max_Heap: #transformar atributo issues em No_Usuario em heap -> extrai max de cada usuario e do heap global, tira se sucesso, do contrario insere de volta
  def __init__(self):
    self.issues = [0]
    self.tam = 0

  def pai(self, indice):
    return floor(i/2)

  def esquerda(self, indice):
    return 2*i

  def direita(self, indice):
    return 2*i + 1

  def troca(self, indiceA, indiceB):
    aux = self.issues[indiceA]
    self.issues[indiceA] = self.issues[indiceB]
    self.issues[indiceB] = aux

  def max_heapify(self, indice):
    esq = self.esquerda(indice)
    dir = self.direita(indice)
    maior = indice

    if esq <= self.tam and self.issues[esq] > self.issues[maior]:
      maior = esq
    if dir <= self.tam and self.issues[dir] > self.issues[maior]:
      maior = dir

    if maior != indice:
      self.troca(indice, maior)
      self.max_heapify(maior)

  def constroi_max_heap(self, issues):
    tamanho_lista = len(issues)
    self.tam = tamanho_lista
    for i in range(floor(tamanho_lista/2),0,-1):
      self.max_heapify(i)

  def heap_max(self):
    return self.issues[1]

  def extraiMax(self):
    if self.tam < 1:
      return -1

    max = self.issues[1]
    self.issues[1] = self.issues[self.tam]
    self.tam-=1
    self.max_heapify(1)
    return max

  def aumentaChave(self,indice,chave):
    if key < self.issues[indice]:
      return -1

    self.issues[indice] = chave
    p = self.pai(indice)
    while i > 1 and self.issues[p] < self.issues[i]:
      self.troca(i, p)
      i = p


  def insert(self, chave):
    self.tam+=1
    n = self.tam
    self.issues[n] = -inf
    self.aumentaChave(n,chave)


## Grafo

In [ ]:
#matriz de adjacência, nós se conectam pela quantidade de issues em ambos. Se 2 usuarios participaram de 3 issues ao mesmo tempo, eles tem 3 conexoes
# a aresta pode ser rotulada com id do repo, visualmente pode ser uma cor
class Grafo:
  def __init__(self, lista_nos):
    self.lista_nos = lista_nos
    self.matrizAdjacencia = []
    inicializaMatrizAdjacencia()

  def inicializaMatrizAdjacencia(self):
    for i in range(len(self.lista_nos)):
      for j in range(len(self.lista_nos)):
        self.matrizAdjacencia[i][j] = 0

  def criar_aresta(self): #não esquecer de add peso
    lista_issues = set()
    for i in self.lista_nos():
      for j in i.issues:
        lista_issues.add(j.id_issue)

    h = Max_Heap()
    for i range(len(lista_issues)):
      h.insert(lista_issues[i])

    h.constroi_max_heap(lista_issues)

    while h.tam > 0:
      m = h.extraiMax()

      for i in lista_nos:
        for j in i.issues:
          if m == j.id_issue:
            #marcar usuarios
            #





  def vizinhos(self):
    pass
  def dfs(self):
    pass
  def brandes(self):
    pass

## AVL

In [ ]:
class No_AVL:
  def __init__(self, esq=None, dir=None, task:Task=None):
    self.esq = esq
    self.dir = dir
    self.fator_balanceamento = 0
    self.altura = 0
    self.conteudo = task

In [ ]:
class AVL:
  def __init__(self): #a árvore representa apenas um punhado de nós ligados, ela em si só precisa das operações para manipular a estrutura de si mesma e pesquisar
      pass
#===============================================================================
# ***************************** MAX ********************************************
#===============================================================================
  def MAX(self, h_esq, h_dir): #verifica qual a maior altura, se é do filho da direita ou da esquerda
    if h_dir >= h_esq:
      return h_dir

    else:
      return h_esq

#===============================================================================
# ***************************** GET_ALTURA *************************************
#===============================================================================
  def getAltura(self, raiz:No_AVL):
    #obtém altura da árvore
    #ponderei fortemente em este ser um atributo da árvore
    if raiz == None:
      return -1
    else:
      return raiz.altura

#===============================================================================
# ***************************** CALCULA_ALTURA *********************************
#===============================================================================
  def calculaAltura(self, raiz:No_AVL):
    #calcula altura do nó corrente
    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)

    return 1 + self.MAX(h_esq, h_dir)

#===============================================================================
# ************************** CALCULA_BALANCEAMENTO *****************************
#===============================================================================
  def calculaFatorBalanceamento(self, raiz:No_AVL):
    #calcula fator de balanceamento do nó
    if raiz == None:
      return

    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)
    raiz.altura = self.calculaAltura(raiz)

    raiz.fator_balanceamento = h_esq - h_dir

#===============================================================================
# ***************************** ROTACAO_ESQUERDA *******************************
#===============================================================================
  def rotacaoESQUERDA(self, raiz:No_AVL):
    n_aux = raiz.dir #no auxiliar que vai virar nova raiz

    if n_aux == None:
      return raiz #não é possível rotacionar para esquerda sem filho da direita, não tem com quem trocar

    t_aux = n_aux.esq #outro no auxiliar, carrega o filho da raiz na rotação

    n_aux.esq = raiz
    raiz.dir = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** ROTACAO_DIREITA ********************************
#===============================================================================
  def rotacaoDIREITA(self,raiz:No_AVL):
    n_aux = raiz.esq

    if n_aux == None:
      return raiz #não é possível rotacionar para direita sem filho da direita, não tem com quem trocar

    t_aux = n_aux.dir

    n_aux.dir = raiz
    raiz.esq = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** BALANCEAR **************************************
#===============================================================================
  def balancear(self, raiz:No_AVL):
    self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================
  def insert(self, raiz:No_AVL, task:Task):
    if raiz == None: #se raiz vazia/folha -> inserimos
      novo_no = No_AVL(None, None, task)
      self.calculaFatorBalanceamento(novo_no)
      return novo_no

    #travessia usual em árvores binárias: se chaveBusca < chave do Nó vai p/ esqquerda
    if task['task_id'] < raiz.conteudo['task_id']:
      raiz.esq = self.insert(raiz.esq, task)

    #Do contrário, vai para direita
    else:
      raiz.dir = self.insert(raiz.dir, task)

    #Após percorrer toda a estrutura, verificar se árvore obedece propriedade da
    #AVL, do contrário, rebalancear
    return self.balancear(raiz)

#===============================================================================
# ***************************** ANTECESSOR *************************************
#===============================================================================
  def antecessor(self, raiz:No_AVL):
    #maior filho da subárvore da esquerda (o mais a direita da esquerda)
    atual = raiz
    while atual.dir != None:
      atual = atual.dir
    return atual

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, raiz:No_AVL, chaveBusca):
    if raiz == None: # nada para remover
      return raiz

    #travessia usual em árvore binária
    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.delete(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.delete(raiz.dir, chaveBusca)

    else:
      #se não tem filho à esquerda ->
      #pai da raiz passa a olhar para filho da direita
      if raiz.esq == None:
        aux = raiz.dir
        raiz = None
        return aux

      #se m]ao tem filho à direita ->
      #pai da raiz passa a aponta para o filho da esquerda
      elif raiz.dir == None:
        aux = raiz.esq
        raiz = None
        return aux

      #caso tenha ambos os filhos, calcula antecessor e este será o novo apontado
      #pela raiz
      aux = self.antecessor(raiz.esq)
      raiz.conteudo = aux.conteudo
      raiz.esq = self.delete(raiz.esq, aux.conteudo['task_id'])

    #verifica se propriedade da avl é respeita e corrige se necessário
    return self.balancear(raiz)

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, raiz:No_AVL, chaveBusca):
    #uma busca usual numa árvore binária
    #se chave de entrada menor que chave do nó, joga para esquerda
    #do contrário direita
    #se chave busca for igual chave nó, ou não existir chave na árvore (None)
    #retorna o Nó
    if raiz == None or raiz.conteudo['task_id'] == chaveBusca:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      return self.search(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      return self.search(raiz.dir, chaveBusca)

#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self, raiz:No_AVL, chaveBuscaA, chaveBuscaB, output:list):
    if raiz == None:
      return output

    if chaveBuscaA < raiz.conteudo['task_id']:
      self.searchinterval(raiz.esq, chaveBuscaA, chaveBuscaB, output)

    #diferença em relação a search é essa condição que verifica se a chave do nó
    #está entre as chaves de entrada
    if chaveBuscaA <= raiz.conteudo['task_id'] <= chaveBuscaB:
      output.append(raiz)

    if chaveBuscaB > raiz.conteudo['task_id']:
      self.searchinterval(raiz.dir, chaveBuscaA, chaveBuscaB, output)

    return output



# Análise